In [ ]:
from robovast.common.analysis import read_output_files, read_output_csv

# DATA_DIR is injected by the robovast analysis runner.
DATA_DIR = ''

traj = read_output_files(DATA_DIR, lambda run_dir: read_output_csv(run_dir, "trajectory.csv"))
metrics = read_output_files(DATA_DIR, lambda run_dir: read_output_csv(run_dir, "metrics.csv"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from robovast.common.campaign_data import read_test_result

PAD_RADIUS = 0.5

def _run_dir(config, run):
    """Resolve a run directory from DATA_DIR (run-, config- or campaign-level)."""
    base = Path(DATA_DIR)
    for cand in (base, base / str(run), base / str(config) / str(run)):
        if (cand / "test.xml").exists():
            return cand
    return base

def outcome_of(run, config):
    # Outcome is derived here (not stored in metrics.csv): pass/fail from test.xml.
    try:
        return 'landed' if read_test_result(_run_dir(config, run))['success'] else 'crashed'
    except Exception:
        return 'unknown'

# Static view per run: descent path (x-z) + tilt over time.
# observed=True: config/run are categorical; only iterate combinations present.
for (config, run), g in traj.groupby(['config', 'run'], observed=True):
    g = g.sort_values('t')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(g['x'], g['z'], '-b')
    ax1.plot(g['x'].iloc[0], g['z'].iloc[0], 'go', label='start')
    ax1.plot(g['x'].iloc[-1], g['z'].iloc[-1], 'rx', label='end')
    ax1.plot([-PAD_RADIUS, PAD_RADIUS], [0, 0], 'g-', lw=4, label='pad')
    ax1.set_xlabel('x [m]'); ax1.set_ylabel('z [m]')
    ax1.set_title(f"{config} run {run}: {outcome_of(run, config)}")
    ax1.legend()
    ax2.plot(g['t'], np.degrees(g['tilt']))
    ax2.axhline(40, color='r', ls='--', lw=0.7); ax2.axhline(-40, color='r', ls='--', lw=0.7)
    ax2.set_xlabel('t [s]'); ax2.set_ylabel('tilt [deg]'); ax2.set_title('tilt')
    plt.tight_layout(); plt.show()

In [ ]:
import base64
from pathlib import Path
from IPython.display import HTML

# The descent animation is rendered by the DescentVideo postprocessing plugin
# (analysis/render.py) into descent.gif per run; embed it as an <img> (the GUI
# renders the cell's HTML output).
gifs = sorted(Path(DATA_DIR or '.').rglob('descent.gif'))
if gifs:
    data = base64.b64encode(gifs[0].read_bytes()).decode()
    out = HTML(f'<img src="data:image/gif;base64,{data}" alt="descent"/>')
else:
    out = HTML('<p>No descent.gif found — run postprocessing '
               '(vast eval gui / vast results postprocess).</p>')
out